In [ ]:
import sys

In [ ]:
!{sys.executable} -m pip install interpax

In [ ]:
sys.path.insert(0, "/home/smanzini/ptauser/work/discovery_run/discovery/src")
import importlib
importlib.reload(importlib.import_module("discovery"))

import discovery as ds
import deterministic_eccentric as det_ecc
#import discovery.samplers.numpyro as ds_numpyro
import os
import numpy as np
import math
import scipy.linalg as sl
import matplotlib.pyplot as plt

import copy

import corner
import json
import pickle
import glob



from scipy import signal

import astropy.units as u
import astropy.constants as astropy_const
import astropy.cosmology.units as cu
from astropy.cosmology import LambdaCDM
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
from jax import jit
from jax.experimental.ode import odeint
from jax.lax import stop_gradient
import jax.scipy.special as jsp
from jax import vmap

from jax.scipy.stats import norm
import re
import h5py

day = 24*3600
yr = 365.25*day
f_yr = 1/yr

Msun = astropy_const.M_sun.to(u.kg).value
c = astropy_const.c.to(u.m / u.s).value
G = astropy_const.G.to(u.m**3 / (u.kg * u.s**2)).value
MGsunsec = Msun*G / c**3
kpc = 1.0* astropy_const.kpc.to(u.m).value/c

%load_ext autoreload
%autoreload



In [ ]:
## get params
with open('source_params_3nHz_e075.json', 'r') as f:
    params = json.load(f)

#cos_gwtheta = params['cos_gwtheta']
#gwphi = params['gwphi']
#raj = 12 h 27 m
#dec = 12 deg 43'
ra_deg  = 187.7059   # degrees
dec_deg =  12.3911   # degrees

gwphi   = np.deg2rad(ra_deg)          # azimuthal angle
gwtheta = np.pi/2 - np.deg2rad(dec_deg)  # polar angle (colatitude)
cos_gwtheta = np.cos(gwtheta)
nu = params['nu']
xi0 = params['xi0']
xi0p = params['xi0p']
gamma0 = params['gamma0']
pol = params['pol']
cos_inc = params['cos_inc']
e0 = params['e0']
log10_mc = 8.83#params['log10_mc']
M = nu**(-3/5)*10**log10_mc
log10_M = np.log10(M)
log10_Forb = params['log10_Forb']
log10_dist = np.log10(16.4)#params['log10_dist']
print(log10_Forb)
cadence = params['cadence']

psrdist = params['psrdist']
mu_psrdist = psrdist
psrdist_sigma = params['psrdist_sigma']
Tobs = params['Tobs']
t = np.arange(0, Tobs*yr, cadence*day)

sigma_toas = params['sigma_toas']


log10_Acrn = params['log10_A_rn']
log10_Arut =np.log10((10**log10_Acrn)*yr/(2*np.sqrt(3)*np.pi) )
gamma_crn = params['gamma_rn']

## pulsars data

psrname = np.array(['J0613-0200', 'J1012+5307', 'J1600-3053', 'J1713+0747', 'J1744-1134', 'J1909-3744'])
print(cos_gwtheta)
print(gwphi)

print(log10_M)
print(log10_mc)

In [ ]:
feathers_path = 'EPTA_feather_DR2new/'

s_dsfiles = np.sort(os.listdir(feathers_path))
print(s_dsfiles)
d_psrs = [ds.pulsar.Pulsar.read_feather(feathers_path + f'{psrfile}') for psrfile in s_dsfiles]
print(d_psrs)

In [ ]:
timedelay = det_ecc.make_delay_eccentric()
cw_common = ['cw_cos_gwtheta', 'cw_gwphi', 'cw_log10_M', 'cw_nu',
             'cw_log10_dist', 'cw_log10_Forb', 'cw_cos_inc', 'cw_psi', 'cw_gamma0',
             'cw_xi0','cw_e0']

def injection_EPTA_DR2new(psrs, crn_components = 30):

    pslmodels = []
    tspan = ds.getspan(psrs)
    for p in psrs:
        tspan = ds.getspan(p)
        #print(len(p.toaerrs))
        #fixed_noise = matrix.NoiseMatrix1D_novar(p.toaerrs**2)

        model = [p.residuals, ds.makenoise_measurement(p, p.noisedict, tnequad = True), ds.makegp_timing(p, svd=True, variance =1e-40), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
        #model = [p.residuals, fixed_noise, ds.makegp_timing(p, svd=True, variance =1e-40), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
         
        model.append(ds.makegp_fourier(p, ds.powerlaw, crn_components, T=tspan, name='crn', common=['crn_log10_A', 'crn_gamma']))
        pslmodels.append(ds.PulsarLikelihood(model))

    tspan = ds.getspan(d_psrs)
    t0 = ds.getstart(d_psrs)
    return ds.GlobalLikelihood(psls = pslmodels)

injection = injection_EPTA_DR2new(d_psrs)
print(injection.logL.params)


In [ ]:
for i in range(len(d_psrs)):
    plt.figure(figsize=(10, 5))
    plt.title(d_psrs[i].name)
    plt.errorbar(d_psrs[i].toas, d_psrs[i].residuals, yerr=d_psrs[i].toaerrs, marker='o', ls='')
    plt.show()

In [ ]:
params = injection.logL.params
params_to_sample = dict(zip(params, np.ones_like(params)))
rng = np.random.default_rng(seed=5)
psrdists = {}
for p in d_psrs:
    psrdists.update({p.name:p.pdist[0]})
print(psrdists)

for param in params_to_sample:
    if '_cw_psrdist' in param:
        # Extract pulsar name
        pulsar_name = param.split('_cw_psrdist')[0]
        
        # Get distance
        if pulsar_name in psrdists:
            params_to_sample[param] = psrdists[pulsar_name]
        else:
            params_to_sample[param] = None  # Still missing

    if '_cw_xi0p' in param:
        # Extract pulsar name
        pulsar_name = param.split('_cw_xi0p')[0]
        
        # Get distance
        if pulsar_name in psrdists:
            params_to_sample[param] = rng.uniform(-np.pi, np.pi)
        else:
            params_to_sample[param] = None  # Still missing
    
print(params_to_sample)


In [ ]:
params_to_sample['cw_cos_gwtheta'] = cos_gwtheta
params_to_sample['cw_cos_inc'] = cos_inc
params_to_sample['cw_e0'] = e0
params_to_sample['cw_gamma0'] = gamma0
params_to_sample['cw_gwphi'] = gwphi
params_to_sample['cw_log10_Forb'] = log10_Forb
params_to_sample['cw_log10_M'] = log10_M
params_to_sample['cw_log10_dist'] = log10_dist
params_to_sample['cw_nu'] = nu
params_to_sample['cw_psi'] = pol
params_to_sample['cw_xi0'] = xi0


params_to_sample['crn_gamma'] = gamma_crn
params_to_sample['crn_log10_A'] = log10_Acrn

print(params_to_sample)
pars_cgw = list(params_to_sample.values())[-11:]
print(pars_cgw)

In [ ]:
key = jax.random.PRNGKey(290699)
key, injected_res = injection.sample(key, params_to_sample)


In [ ]:
true_residuals = []
for i in range(len(d_psrs)):
    print(d_psrs[i].name + '_cw_xi0p')
    true_residuals_p0 = eccjaxPTA_stas.full_residuals_fast(d_psrs[i].toas, np.array(d_psrs[i].pos), d_psrs[i].pdist[0], params_to_sample['cw_cos_gwtheta'], params_to_sample['cw_gwphi'], params_to_sample['cw_log10_M'], params_to_sample['cw_nu'],
                                   params_to_sample['cw_log10_dist'], params_to_sample['cw_log10_Forb'], params_to_sample['cw_cos_inc'], params_to_sample['cw_psi'], params_to_sample['cw_gamma0'],
                                   params_to_sample['cw_xi0'],params_to_sample[str(d_psrs[i].name + '_cw_xi0p')] , params_to_sample['cw_e0'], 0)
    true_residuals.append(true_residuals_p0)
    plt.plot(d_psrs[i].toas, injected_res[i])
    plt.plot(d_psrs[i].toas, true_residuals_p0)
    
    plt.show()

In [ ]:
#  save in the psrs residuals
for ii, psr in enumerate(d_psrs):
    psr.residuals = np.array(injected_res[ii]).squeeze()

def fit_EPTA_DR2new(psrs, crn_components = 30):

    pslmodels = []
    tspan = ds.getspan(psrs)
    for p in psrs:
        tspan = ds.getspan(p)
        #print(len(p.toaerrs))
        #fixed_noise = matrix.NoiseMatrix1D_novar(p.toaerrs**2)

        model = [p.residuals, ds.makenoise_measurement(p, p.noisedict, tnequad = True), ds.makegp_timing(p, svd=True), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
        #model = [p.residuals, fixed_noise, ds.makegp_timing(p, svd=True, variance =1e-40), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
         
        model.append(ds.makegp_fourier(p, ds.powerlaw, crn_components, T=tspan, name='crn', common=['crn_log10_A', 'crn_gamma']))
        pslmodels.append(ds.PulsarLikelihood(model))

    tspan = ds.getspan(psrs)
    t0 = ds.getstart(psrs)
    return ds.GlobalLikelihood(psls = pslmodels)


In [ ]:

dirpath = '/home/smanzini/ptauser/work/WN+CRN+CGW_e0.75_3nHz_feathers_realWN_SNR8_seed290699/'
'''

if not os.path.exists(dirpath):
        os.makedirs(dirpath)

for psr in d_psrs:
    ds.pulsar.Pulsar.save_feather(psr, dirpath + psr.name, noisedict=psr.noisedict)
'''

In [ ]:

with open(dirpath + '/injected_params.json', 'w') as f:
    json.dump(params_to_sample, f, indent=4)


In [ ]:
#dirpath = '/home/smanzini/ptauser/work/WN+CRN+CGW_e0.75_3nHz_feathers_realWN_6temps/'
with open(dirpath + 'injected_params.json', 'r') as f:
    injected_params = jnp.array(list(json.load(f).values()))
print(injected_params)

with open(dirpath + 'injected_params.json', 'r') as f:
    injected_dict = json.load(f)

with open(dirpath + 'prior_dict.json', 'r') as f:
    priors_all = json.load(f)


In [ ]:
feathers_path = '/home/smanzini/ptauser/work/WN+CRN+CGW_e0.75_3nHz_feathers_realWN_SNR8_seed290699/'

def fit_EPTA_DR2new(psrs, crn_components = 30):

    pslmodels = []
    tspan = ds.getspan(psrs)
    for p in psrs:
        tspan = ds.getspan(p)
        print(len(p.toaerrs))
        #fixed_noise = matrix.NoiseMatrix1D_novar(p.toaerrs**2)

        model = [p.residuals, ds.makenoise_measurement(p, p.noisedict, tnequad = True), ds.makegp_timing(p, svd=True), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
        
         
        model.append(ds.makegp_fourier(p, ds.powerlaw, crn_components, T=tspan, name='crn', common=['crn_log10_A', 'crn_gamma']))
        pslmodels.append(ds.PulsarLikelihood(model))

    tspan = ds.getspan(psrs)
    t0 = ds.getstart(psrs)
    return ds.GlobalLikelihood(psls = pslmodels)

ds_files = np.sort([f for f in os.listdir(feathers_path) if f.startswith('J')])
print(ds_files)
d_psrs_read = [ds.pulsar.Pulsar.read_feather(feathers_path + f'{psrfile}') for psrfile in ds_files]

fit = fit_EPTA_DR2new(d_psrs)
print(fit.logL.params)

for i in range(len(d_psrs_read)):
    #plt.plot(d_psrs[i].toas, d_psrs[i].residuals)
    #plt.plot(d_psrs_noise[i].toas, d_psrs_noise[i].residuals, alpha = 0.5, ls = '--')
    plt.plot(d_psrs[i].toas, d_psrs[i].residuals-true_residuals[i], alpha = 0.5, ls = '--')
    plt.plot(d_psrs_read[i].toas, d_psrs_read[i].residuals-true_residuals[i], alpha = 0.5, ls = '--')
    plt.show()


#print(injected_params)
likelihood_injected = fit.logL(dict(zip(params_to_sample, injected_params)))
print(d_psrs_read[0].noisedict)
print(likelihood_injected)


In [ ]:
params_signal = dict(zip(fit.logL.params, injected_params))
params_noise  = params_signal.copy()
params_noise['cw_log10_dist'] = 10.0  # push distance to infinity → zero signal

logL_signal = float(fit.logL(params_signal))
logL_noise  = float(fit.logL(params_noise))

delta_logL = logL_signal - logL_noise
SNR = np.sqrt(2 * abs(delta_logL))

print(f"logL_signal: {logL_signal:.2f}")
print(f"logL_noise:  {logL_noise:.2f}")
print(f"delta_logL:  {delta_logL:.2f}")
print(f"SNR:         {SNR:.2f}")

In [ ]:
# Split sampled vs constant
params_sampled = [name for name in priors_all.keys() if priors_all[name]['dist'] != 'constant']
params_constant = {name: priors_all[name]['value'] for name in priors_all.keys() if priors_all[name]['dist'] == 'constant'}

params_to_sample = np.array(params_sampled)
n_params = len(params_sampled)

print(f"Sampled ({n_params}): {params_sampled}")
print(f"Constant: {params_constant}")

# Separate normal and uniform among sampled only
normal_indices = []
uniform_indices = []
normal_mus = []
normal_sigmas = []
uniform_lbs = []
uniform_ubs = []

for i, name in enumerate(params_sampled):
    prior = priors_all[name]
    if prior['dist'] == 'normal':
        normal_indices.append(i)
        normal_mus.append(prior['mu'])
        normal_sigmas.append(prior['sigma'])
    elif prior['dist'] == 'uniform':
        uniform_indices.append(i)
        uniform_lbs.append(prior['min'])
        uniform_ubs.append(prior['max'])
    # constant is skipped — not sampled

normal_mus = jnp.array(normal_mus)
normal_sigmas = jnp.array(normal_sigmas)
uniform_lbs = jnp.array(uniform_lbs)
uniform_ubs = jnp.array(uniform_ubs)
eps = 1e-10
normal_indices = np.array(normal_indices)
uniform_indices = np.array(uniform_indices)

print(f"Normal indices: {normal_indices}")
print(f"Uniform indices: {uniform_indices}")
print(f"Normal + Uniform = {len(normal_indices) + len(uniform_indices)} (should equal {n_params})")

from jax.scipy.special import ndtr, ndtri
@jax.jit
def prior_transform_single(u):
    """Transform [0,1]^n -> physical parameters (sampled dims only)"""
    x = jnp.zeros(n_params)
    if len(normal_indices) > 0:
        u_normal = u[normal_indices]
        x_normal = normal_mus + jnp.maximum(normal_sigmas, eps) * ndtri(u_normal)
        x = x.at[normal_indices].set(x_normal)
    if len(uniform_indices) > 0:
        u_uniform = u[uniform_indices]
        x_uniform = uniform_lbs + u_uniform * (uniform_ubs - uniform_lbs)
        x = x.at[uniform_indices].set(x_uniform)
    return x

In [ ]:
param_names = list(priors_all.keys())

# Pulsar distances (first 6)
psrdist_idx = [param_names.index(k) for k in param_names if 'psrdist' in k]

# xi0p (last 6)
xi0p_idx = [param_names.index(k) for k in param_names if 'xi0p' in k]

# CGW params in desired order
cgw_keys = [
    'cw_cos_gwtheta', 'cw_gwphi', 'cw_log10_M', 'cw_log10_dist',
    'cw_log10_Forb', 'cw_cos_inc', 'cw_psi', 'cw_gamma0',
    'cw_e0', 'cw_xi0', 'cw_nu'
]
cgw_idx = [param_names.index(k) for k in cgw_keys]

new_order = psrdist_idx + cgw_idx + xi0p_idx
print(new_order)  # sanity check: should be a permutation of 0..22

print(len(param_names))
print(priors_all)

In [ ]:
outdir = '/home/smanzini/ptauser/work/WN+CRN+CGW_e0.75_3nHz_feathers_realWN_SNR8_seed290699/'
chain_files = sorted(glob.glob(outdir + '/*.h5'))
print(f"Found {len(chain_files)} chain files: {[f.split('/')[-1] for f in chain_files]}")
all_p  = []
all_ll = []
ntemps  = 5
nparams = len(param_names)

for fpath in chain_files:
    with h5py.File(fpath, 'r') as hf:
        for ekey in sorted(hf.keys()):
            p  = hf[f'{ekey}/samples/p'][()]   # (nwalkers*ntemps, nparams, niters)
            ll = hf[f'{ekey}/samples/ll'][()]  # (nwalkers*ntemps, 1, niters)
            nsteps   = p.shape[-1]
            nwalkers = int(p.shape[0] / ntemps)
            all_p.append(p.reshape(nwalkers, ntemps, nparams, nsteps))
            all_ll.append(ll[:, 0, :].reshape(nwalkers, ntemps, nsteps))

#del all_p[0]
#del all_ll[0]

# concatenate along niters axis
samples_unit = np.concatenate(all_p,  axis=3)  # (nwalkers, ntemps, nparams, niters)
loglike_flat = np.concatenate(all_ll, axis=2)  # (nwalkers, ntemps, niters)

nwalkers = samples_unit.shape[0]
ntemps   = samples_unit.shape[1]
nparams  = samples_unit.shape[2]
niters   = samples_unit.shape[3]
print(f"nwalkers={nwalkers}, ntemps={ntemps}, nparams={nparams}, niters={niters}")

# Log-likelihood trace — cold chain
burn_in = int(2e3)
thin    = int(5)
# transpose to (niters, ntemps, nwalkers, nparams)
samples_unit_r = np.transpose(samples_unit, (3, 1, 0, 2))[burn_in::thin,:, :, :]
loglike_r      = np.transpose(loglike_flat, (2, 1, 0))[burn_in::thin, :, :]
print(f"samples_unit_r: {samples_unit_r.shape}")  # (niters, ntemps, nwalkers, nparams)
print(f"loglike_r:      {loglike_r.shape}")        # (niters, ntemps, nwalkers)

# reorder parameters to [psrdist | cgw | xi0p]
#samples_unit_r = samples_unit_r[:, :, :, :]

# sanity check unit cube
print(f"samples min={samples_unit_r.min():.4f}, max={samples_unit_r.max():.4f}  (should be in [0,1])")
print(f"cold chain mean ll: {loglike_r[:, 0, :].mean():.2f}")
print(f"hot  chain mean ll: {loglike_r[:, -1, :].mean():.2f}")
print(f"cold chain max  ll: {loglike_r[:, 0, :].max():.2f}")
print(f"injected ll: {float(likelihood_injected):.2f}")

# transform to physical
transform_chains = jax.vmap(jax.vmap(jax.vmap(prior_transform_single)))
chains_physical = np.array(transform_chains(jnp.array(samples_unit_r)))
print(f"chains_physical shape: {chains_physical.shape}")  # (niters, ntemps, nwalkers, nparams)


fig, ax = plt.subplots(figsize=(12, 3))
for nw in range(0, nwalkers):
    ax.plot(loglike_r[:, 0,nw], alpha=0.5, lw=0.5, color='steelblue')
    ax.plot(loglike_r[:, -1,nw], alpha=0.5, lw=0.5, color='r')

ax.axhline(likelihood_injected)
ax.axhline(logL_noise)
ax.axvline(burn_in, color='red', lw=1.5, linestyle='--', label='burn-in')
ax.set_title('log-likelihood — cold chain')
ax.set_xlabel('iteration')
ax.legend()
plt.tight_layout()
plt.show()

'''
# Trace plots — cold chain
for i, name in enumerate(params_sampled):
    fig, ax = plt.subplots(figsize=(12, 3))
    for nw in range(nwalkers):
        ax.plot(chains_physical[:, 0, nw, i], lw=0.5, color='steelblue')
        #ax.plot(chains_physical[:, 1, nw, i], alpha=0.2, lw=0.5, color='red')
    ax.axvline(burn_in, color='red', lw=1.5, linestyle='--', label='burn-in')
    inj = injected_dict.get(name, None)
    if inj is not None:
        ax.axhline(float(inj), color='orange', lw=1.5, label=f'injected={float(inj):.4f}')
    ax.set_title(name)
    ax.set_xlabel('iteration')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
'''


#(240, 5, 250, 23)
# Corner plot — cold chain after burn-in
cold_flat = chains_physical[:, 0, :, :].reshape(-1, nparams)
next_flat = chains_physical[:, 1, :, :].reshape(-1, nparams)
print(f"Samples for corner: {cold_flat.shape}")
for i in range(len(params_to_sample)):
    plt.plot(cold_flat[:, i], alpha=0.5, lw=0.5, color='steelblue')
    #plt.plot(next_flat[:, i], alpha=0.5, lw=0.5, color='red')
    plt.show()  
    
cw_params  = [n for n in params_sampled if 'cw_' in n and 'psrdist' not in n and 'xi0p' not in n]
cw_indices = [params_sampled.index(n) for n in cw_params]
cw_truths  = [float(injected_dict[n]) for n in cw_params]

fig = corner.corner(
    cold_flat[:, cw_indices],
    labels=cw_params,
    truths=cw_truths,
    truth_color='orange',
    show_titles=True,
    plot_datapoints = False,
    fill_contours = False,
    title_kwargs={'fontsize': 8},
    label_kwargs={'fontsize': 8},
    color = 'blue',
    bins=30
)

corner.corner(
    next_flat[:, cw_indices],
    labels=cw_params,
    truths=cw_truths,
    truth_color='orange',
    show_titles=True,
    title_kwargs={'fontsize': 8},
    label_kwargs={'fontsize': 8},
    plot_datapoints = False,
    fill_contours = False,
    color = 'red',
    bins=30,
    fig = fig
)
plt.suptitle('CW parameters — cold chain', y=1.01)
#plt.savefig(outdir + '/corner_cw.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import gc
del samples_unit, samples_unit_r, chains_physical, loglike_flat, all_p, all_ll
gc.collect()

In [ ]:
d_psrs_noise = copy.deepcopy(d_psrs)
def inject_noise(psrs, crn_components = 30):

    pslmodels = []
    tspan = ds.getspan(psrs)
    for p in psrs:
        tspan = ds.getspan(p)
        #print(len(p.toaerrs))
        #fixed_noise = matrix.NoiseMatrix1D_novar(p.toaerrs**2)

        model = [p.residuals, ds.makenoise_measurement(p, p.noisedict, tnequad = True), ds.makegp_timing(p, svd=True, variance = 1e-40)]#, ds.makedelay(p, timedelay, common=cw_common, name='cw')]
        #model = [p.residuals, fixed_noise, ds.makegp_timing(p, svd=True, variance =1e-40), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
         
        model.append(ds.makegp_fourier(p, ds.powerlaw, crn_components, T=tspan, name='crn', common=['crn_log10_A', 'crn_gamma']))
        pslmodels.append(ds.PulsarLikelihood(model))

    tspan = ds.getspan(psrs)
    t0 = ds.getstart(psrs)
    return ds.GlobalLikelihood(psls = pslmodels)

noise_inj = inject_noise(d_psrs_noise)
noise_params = inject_noise(d_psrs_noise).logL.params

In [ ]:

noise_params_dict = dict(zip(noise_params, np.ones_like(noise_params)))
noise_params_dict['crn_gamma'] = gamma_crn
noise_params_dict['crn_log10_A'] = log10_Acrn

key = jax.random.PRNGKey(290699)
key, injected_noise = noise_inj.sample(key, noise_params_dict)

for ii, psr in enumerate(d_psrs_noise):
    psr.residuals = np.array(injected_noise[ii]).squeeze()

In [ ]:
'''for i in range(len(d_psrs)):
    plt.plot(d_psrs[i].toas, d_psrs[i].residuals-d_psrs_noise[i].residuals, label='signal+noise')
    plt.plot(d_psrs_noise[i].toas, true_residuals[i], label='noise only', alpha=0.7)
    plt.legend()
    plt.show()
'''

In [ ]:

dirpath_noise = '/home/smanzini/ptauser/work/WN+CRN_e0.75_3nHz_feathers_realWN_SNR8/'


if not os.path.exists(dirpath_noise):
        os.makedirs(dirpath_noise)

for psr in d_psrs_noise:
    ds.pulsar.Pulsar.save_feather(psr, dirpath_noise + psr.name, noisedict=psr.noisedict)


with open(dirpath_noise + '/injected_params.json', 'w') as f:
    json.dump(noise_params_dict, f, indent=4)
print(noise_params_dict)

In [ ]:
def fit_noise(psrs, crn_components = 30):

    pslmodels = []
    tspan = ds.getspan(psrs)
    for p in psrs:
        tspan = ds.getspan(p)
        #print(len(p.toaerrs))
        #fixed_noise = matrix.NoiseMatrix1D_novar(p.toaerrs**2)

        model = [p.residuals, ds.makenoise_measurement(p, p.noisedict, tnequad = True), ds.makegp_timing(p, svd=True)]#, ds.makedelay(p, timedelay, common=cw_common, name='cw')]
        #model = [p.residuals, fixed_noise, ds.makegp_timing(p, svd=True, variance =1e-40), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
         
        model.append(ds.makegp_fourier(p, ds.powerlaw, crn_components, T=tspan, name='crn', common=['crn_log10_A', 'crn_gamma']))
        pslmodels.append(ds.PulsarLikelihood(model))

    tspan = ds.getspan(psrs)
    t0 = ds.getstart(psrs)
    return ds.GlobalLikelihood(psls = pslmodels)

noise_fit = fit_noise(d_psrs_noise)
logll_nn_C = noise_fit.logL(noise_params_dict)
print(logll_nn_C)

In [ ]:
def fit_noise_cgw(psrs, crn_components = 30):

    pslmodels = []
    tspan = ds.getspan(psrs)
    for p in psrs:
        tspan = ds.getspan(p)
        #print(len(p.toaerrs))
        #fixed_noise = matrix.NoiseMatrix1D_novar(p.toaerrs**2)

        model = [p.residuals, ds.makenoise_measurement(p, p.noisedict, tnequad = True), ds.makegp_timing(p, svd=True), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
        #model = [p.residuals, fixed_noise, ds.makegp_timing(p, svd=True, variance =1e-40), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
         
        model.append(ds.makegp_fourier(p, ds.powerlaw, crn_components, T=tspan, name='crn', common=['crn_log10_A', 'crn_gamma']))
        pslmodels.append(ds.PulsarLikelihood(model))

    tspan = ds.getspan(psrs)
    t0 = ds.getstart(psrs)
    return ds.GlobalLikelihood(psls = pslmodels)

noise_cgw_fit = fit_noise_cgw(d_psrs_noise)
logll_nn_C_2 = noise_cgw_fit.logL(params_noise)
print(logll_nn_C_2)

logll_ns = noise_cgw_fit.logL(params_signal)
print(logll_ns)

In [ ]:
A = logll_ns-logll_nn_C_2
print(A)
B = (logL_signal - logL_noise)
print(B)

SNR_2 = B-A
print(np.sqrt(SNR_2))
n_s = (A+B)/2
print(n_s)
print(np.sqrt(SNR_2 + 2*n_s))
print(np.sqrt(2*B))

In [ ]:
def fit_noise_cgw(psrs, crn_components = 30):

    pslmodels = []
    tspan = ds.getspan(psrs)
    for p in psrs:
        tspan = ds.getspan(p)
        #print(len(p.toaerrs))
        #fixed_noise = matrix.NoiseMatrix1D_novar(p.toaerrs**2)

        model = [p.residuals, ds.makenoise_measurement(p, p.noisedict, tnequad = True), ds.makegp_timing(p, svd=True), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
        #model = [p.residuals, fixed_noise, ds.makegp_timing(p, svd=True, variance =1e-40), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
         
        model.append(ds.makegp_fourier(p, ds.powerlaw, crn_components, T=tspan, name='crn', common=['crn_log10_A', 'crn_gamma']))
        pslmodels.append(ds.PulsarLikelihood(model))

    tspan = ds.getspan(psrs)
    t0 = ds.getstart(psrs)
    return ds.GlobalLikelihood(psls = pslmodels)

noise_cgw_fit = fit_noise_cgw(d_psrs_noise)
logll_nn_C_2 = noise_cgw_fit.logL(params_noise)
print(logll_nn_C_2)